In [1]:
1

1

In [2]:
from safetensors.torch import load_file
import json

# Load the weights
weights = load_file(r"D:\Projects\IMPOLS\pretrained_models\gpt2\model.safetensors")

# See all available keys
for key in weights.keys():
    print(f"{key:60s} {str(weights[key].shape)}")

h.0.attn.c_attn.bias                                         torch.Size([2304])
h.0.attn.c_attn.weight                                       torch.Size([768, 2304])
h.0.attn.c_proj.bias                                         torch.Size([768])
h.0.attn.c_proj.weight                                       torch.Size([768, 768])
h.0.ln_1.bias                                                torch.Size([768])
h.0.ln_1.weight                                              torch.Size([768])
h.0.ln_2.bias                                                torch.Size([768])
h.0.ln_2.weight                                              torch.Size([768])
h.0.mlp.c_fc.bias                                            torch.Size([3072])
h.0.mlp.c_fc.weight                                          torch.Size([768, 3072])
h.0.mlp.c_proj.bias                                          torch.Size([768])
h.0.mlp.c_proj.weight                                        torch.Size([3072, 768])
h.1.attn.c_attn.bias       

In [3]:
#load config
with open(r"D:\Projects\IMPOLS\pretrained_models\gpt2\config.json") as f:
    hf_config = json.load(f)

print(hf_config)

{'activation_function': 'gelu_new', 'architectures': ['GPT2Model'], 'attn_pdrop': 0.1, 'bos_token_id': 50256, 'dtype': 'float32', 'embd_pdrop': 0.1, 'eos_token_id': 50256, 'initializer_range': 0.02, 'layer_norm_epsilon': 1e-05, 'model_type': 'gpt2', 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_inner': None, 'n_layer': 12, 'n_positions': 1024, 'reorder_and_upcast_attn': False, 'resid_pdrop': 0.1, 'scale_attn_by_inverse_layer_idx': False, 'scale_attn_weights': True, 'summary_activation': None, 'summary_first_dropout': 0.1, 'summary_proj_to_labels': True, 'summary_type': 'cls_index', 'summary_use_proj': True, 'task_specific_params': {'text-generation': {'do_sample': True, 'max_length': 50}}, 'transformers_version': '4.57.6', 'use_cache': True, 'vocab_size': 50257}


In [4]:
from safetensors.torch import load_file

hf_weights = load_file(r"D:\Projects\IMPOLS\pretrained_models\gpt2\model.safetensors")

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch → Left: {left.shape}, Right: {right.shape}")
    return nn.Parameter(right.clone().detach().float())

def load_hf_into_my_gpt(model, w):
    model.token_emb.weight = assign(model.token_emb.weight, w["wte.weight"])
    model.pos_emb.weight   = assign(model.pos_emb.weight,   w["wpe.weight"])

    for b in range(cfg["n_layer"]):
        p  = f"h.{b}"
        tb = model.tranformer_block[b]

        # LayerNorms
        tb.layernorm1.scale = assign(tb.layernorm1.scale, w[f"{p}.ln_1.weight"])
        tb.layernorm1.shift = assign(tb.layernorm1.shift, w[f"{p}.ln_1.bias"])
        tb.layernorm2.scale = assign(tb.layernorm2.scale, w[f"{p}.ln_2.weight"])
        tb.layernorm2.shift = assign(tb.layernorm2.shift, w[f"{p}.ln_2.bias"])

        # Q, K, V (split from concatenated HuggingFace format)
        qkv_w = w[f"{p}.attn.c_attn.weight"]  # [768, 2304]
        qkv_b = w[f"{p}.attn.c_attn.bias"]    # [2304]
        q_w, k_w, v_w = torch.split(qkv_w, 768, dim=1)
        q_b, k_b, v_b = torch.split(qkv_b, 768, dim=0)

        tb.mutihead_atten.w_query.weight = assign(tb.mutihead_atten.w_query.weight, q_w.T)
        tb.mutihead_atten.w_key.weight   = assign(tb.mutihead_atten.w_key.weight,   k_w.T)
        tb.mutihead_atten.w_value.weight = assign(tb.mutihead_atten.w_value.weight, v_w.T)
        tb.mutihead_atten.w_query.bias   = assign(tb.mutihead_atten.w_query.bias,   q_b)
        tb.mutihead_atten.w_key.bias     = assign(tb.mutihead_atten.w_key.bias,     k_b)
        tb.mutihead_atten.w_value.bias   = assign(tb.mutihead_atten.w_value.bias,   v_b)

        # Attention output projection
        tb.mutihead_atten.out_proj.weight = assign(
            tb.mutihead_atten.out_proj.weight, w[f"{p}.attn.c_proj.weight"].T)
        tb.mutihead_atten.out_proj.bias   = assign(
            tb.mutihead_atten.out_proj.bias,   w[f"{p}.attn.c_proj.bias"])

        # FeedForward
        tb.feed_forward.layer[0].weight = assign(
            tb.feed_forward.layer[0].weight, w[f"{p}.mlp.c_fc.weight"].T)
        tb.feed_forward.layer[0].bias   = assign(
            tb.feed_forward.layer[0].bias,   w[f"{p}.mlp.c_fc.bias"])
        tb.feed_forward.layer[2].weight = assign(
            tb.feed_forward.layer[2].weight, w[f"{p}.mlp.c_proj.weight"].T)
        tb.feed_forward.layer[2].bias   = assign(
            tb.feed_forward.layer[2].bias,   w[f"{p}.mlp.c_proj.bias"])

    # Final LayerNorm
    model.final_norm.scale = assign(model.final_norm.scale, w["ln_f.weight"])
    model.final_norm.shift = assign(model.final_norm.shift, w["ln_f.bias"])

    # Output head (weight tying)
    model.out_head.weight  = assign(model.out_head.weight,  w["wte.weight"])

    print("✅ Weights loaded successfully!")

In [5]:
from GPT_Model import *

In [6]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,       # set to 0 for inference
    "qkv_bias": True        # GPT-2 uses bias
}



In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gpt_model = GPT(cfg)        # must recreate after fixing cfg!
gpt_model.eval()

load_hf_into_my_gpt(gpt_model, hf_weights)
gpt_model.to(device)



✅ Weights loaded successfully!


GPT(
  (token_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (tranformer_block): Sequential(
    (0): Transfomers(
      (layernorm1): LayerNorm()
      (layernorm2): LayerNorm()
      (mutihead_atten): MultiHeadAttention(
        (w_query): Linear(in_features=768, out_features=768, bias=True)
        (w_key): Linear(in_features=768, out_features=768, bias=True)
        (w_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (feed_forward): FeedForward(
        (layer): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (1): Transfomers(
      (layernorm1): LayerNorm()
      (layernorm2): Laye

In [17]:
def predict(gpt_model=gpt_model,prompt="",next_new_token=5,temp=.7,topk=4):
  #temp+topk implemention to find next token generation
  tokenid=generate_text(
    model=gpt_model,
    ip_token_id = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device),
    max_new_tokens=next_new_token,
    context_size=int(cfg["context_len"]/4),
    temp=temp,
    topk=topk
  )
  return (token_id_to_text(tokenid,tokenizer))

In [ ]:
# Test generate
predict(prompt="Every step move you",next_new_token=7,temp=1,topk=3)

['Every step move you can take to make your life better']

In [23]:
# Test generate
predict(prompt="Every step move you",next_new_token=10,temp=.3,topk=3)

['Every step move you can take to make it happen.\n\n"']

In [9]:
# Test
# prompt    = "Today is monday then tommorow is Tuesday and tommorrow will be "
# input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

# output = generate_text(gpt_model, input_ids, max_new_tokens=10, context_size=cfg["context_len"])
# print(tokenizer.decode(output[0].tolist()))

In [11]:
# Test
# prompt    = "2 plus 2 equal to write the answer afer perfoming the mathmatical calculation"
# input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

# output = generate_text(gpt_model, input_ids, max_new_tokens=50, context_size=cfg["context_len"])
# print(tokenizer.decode(output[0].tolist()))

In [12]:
# Test
prompt    = "Q: What is 2 + 2?\nA"
# input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

# output = generate_text(gpt_model, input_ids, max_new_tokens=50, context_size=cfg["context_len"])
# print(tokenizer.decode(output[0].tolist()))

In [13]:
prompt = "Q: 2 + 2 = ?\nA:"

['Q: 2 + 2 = ?\nA: 2 + 2 = ?\n']
